In [1]:
import pandas as pd
import numpy as np
import robustsharpe as rs

In [2]:
average_model_path = "../result/total_model_average.csv"
equal_strategy_path = "../result/equal_strategy.csv"

In [3]:
grpo_best = pd.read_csv(average_model_path, index_col=0)

In [4]:
grpo_best

,GRPO,PPO,SAC
2019-01-30,-0.023335,0.019726,0.030210
2019-02-28,0.015721,0.009734,0.006705
2019-03-28,-0.008980,-0.008134,-0.010164
2019-04-26,0.002209,0.021928,0.012148
2019-05-24,-0.011915,-0.027820,-0.020592
...,...,...,...
2024-09-20,0.011247,0.001493,0.032385
2024-10-18,0.010479,0.016962,0.004471
2024-11-15,-0.022477,-0.020190,-0.031152
2024-12-16,0.065416,0.078830,0.036831


In [5]:
eqaul = pd.read_csv(equal_strategy_path, index_col=0)

In [21]:
my_strategy_returns = np.array(grpo_best["SAC"])

In [22]:
benchmark_returns = np.array(eqaul["equal_asset_weight"])

In [23]:
returns = np.stack([my_strategy_returns, benchmark_returns], axis=1)  # shape: (T, 2)

In [24]:
# returns = np.stack([benchmark_returns, my_strategy_returns], axis=1)  # shape: (T, 2)

In [25]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,         # shape (T, 2)
#     b_vec=[1, 2, 4, 6],  # 후보 block size
#     alpha=0.05,
#     M=199,                    # bootstrap per test
#     K=1000,                    # pseudo-sequence 생성 횟수
#     T_start= 20
# )


In [26]:
b_vec=[1, 2, 3, 4, 5, 6]

In [27]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,
#     b_vec=[1, 2, 3, 4, 5, 6],  # T=76이므로 6 이상은 피하는게 좋음
#     alpha=0.05,
#     K=300,
#     M=99,
#     T_start=20
# )


In [28]:
SRs, diff, ci, pval, se, d = rs.bootstrap_inference(
    returns=returns,
    block_size=4,     # 논문 추천값 (T=120 기준)
    alpha=0.05,
    M=1000
)

print("Sharpe ratio (벤치마크, 내):", SRs)
print("Sharpe ratio 차이:", diff)
print("95% 신뢰구간:", ci)
print("t-통계량", d)
print("p-value:", pval)

Sharpe ratio (벤치마크, 내): [0.18239849 0.14552373]
Sharpe ratio 차이: -0.03687476221972086
95% 신뢰구간: (np.float64(-0.15382895628895454), np.float64(0.08007943184951283))
t-통계량 0.5572970089586141
p-value: 0.4885114885114885


In [46]:
diff / se

np.float64(-1.5650429769955672)

In [47]:
eqaul["equal_asset_weight"]

2019-01-30    0.054332
2019-02-28    0.016301
2019-03-28   -0.004392
2019-04-26    0.024158
2019-05-24   -0.027220
                ...   
2024-09-20    0.034407
2024-10-18    0.002778
2024-11-15   -0.020486
2024-12-16    0.036143
2024-12-31   -0.022752
Name: equal_asset_weight, Length: 76, dtype: float64